In [ ]:
conda activate seacells_gpu

In [ ]:
import os
os.chdir(path_to_wd)
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import torch
print(torch.cuda.is_available()) 

import numpy as np
import pandas as pd
import scanpy as sc
import SEACells
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import palantir
from tqdm.auto import tqdm
import pickle
from scipy.io import mmread
import scipy.sparse as sp

# Some plotting aestheticsqfree
sns.set_style('ticks')
matplotlib.rcParams['figure.figsize'] = [4, 4]
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['image.cmap'] = 'Spectral_r'

## Load data

In [ ]:
adata = sc.read("Reproducibility/Data/TREKKER/UC_TREKKER_RNA_ATAC_Epithelial.h5ad")

In [ ]:
adata_tmp = adata.copy()
rna_ad = adata_tmp[:, adata_tmp.var.feature_types  == "Gene Expression"].copy()

In [ ]:
# ATAC
data_dir = os.path.expanduser(f"Reproducibility/Results/ArchR/TREKKER_Epithelial/export/")
counts = mmread(data_dir + "peak_counts/counts.mtx")
if sp.issparse(counts) and not sp.isspmatrix_csr(counts):
    counts = counts.tocsr()

cells = pd.read_csv(data_dir + "peak_counts/cells.csv", index_col=0).iloc[:, 0]
peaks = pd.read_csv(data_dir + "peak_counts/peaks.csv", index_col=0)
peaks.index = peaks['seqnames'] + ':' + peaks['start'].astype(str) + '-' + peaks['end'].astype(str)

atac_ad = sc.AnnData(counts.T)
atac_ad.obs_names = cells
atac_ad.var_names = peaks.index
    
atac_ad.X = atac_ad.X.tocsr()

# SVD embeddings
atac_ad.obsm['X_svd'] = pd.read_csv(f'{data_dir}svd.csv', index_col=0).loc[atac_ad.obs_names, :].values

# reorder CB & obs
CB_list = [idx.rsplit("#")[1] for idx in atac_ad.obs_names]
atac_ad.obs_names = CB_list

# Intersection of cells
cells_use = np.intersect1d(rna_ad.obs.index, atac_ad.obs.index)
rna_ad = rna_ad[cells_use, :]
atac_ad = atac_ad[cells_use, :]
atac_ad.obs = rna_ad.obs.copy()
atac_ad.obsm['MultiVI_latent'] = rna_ad.obsm['MultiVI_latent']
atac_ad.obsm['X_umap'] = rna_ad.obsm['X_umap']

In [ ]:
#================================
# Core parameters 
#================================
# recommended choosing one metacell for every 75-100 single-cells
n_cells = len(atac_ad.obs)
n_SEACells = n_cells//75
build_kernel_on = 'MultiVI_latent' # key in ad.obsm to use for computing metacells

## Additional parameters
n_waypoint_eigs = 10           # Number of eigenvalues to consider when initializing metacells
model = SEACells.core.SEACells(atac_ad, 
                  build_kernel_on=build_kernel_on, 
                  use_gpu = False,
                  n_SEACells=n_SEACells, 
                  n_waypoint_eigs=n_waypoint_eigs,
                  convergence_epsilon = 1e-5)
model.construct_kernel_matrix()
M = model.kernel_matrix
model.initialize_archetypes()
model.fit(max_iter=200) 

In [ ]:
1/14 kokomade

In [ ]:
# evaluation
model.plot_convergence()
SEACells.plot.plot_2D(atac_ad, key="X_umap", colour_metacells=False)
SEACells.plot.plot_SEACell_sizes(atac_ad, bins=5)

compactness = SEACells.evaluate.compactness(atac_ad, "MultiVI_latent")
SEACell_purity = SEACells.evaluate.compute_celltype_purity(atac_ad, "clone2")
separation = SEACells.evaluate.separation(atac_ad, "MultiVI_latent", nth_nbr=1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)
sns.boxplot(data=compactness, y="compactness", ax=axes[0]); axes[0].set_title("Compactness"); sns.despine(ax=axes[0])
sns.boxplot(data=SEACell_purity, y="clone2_purity", ax=axes[1]); axes[1].set_title("clone2 Purity"); sns.despine(ax=axes[1])
sns.boxplot(data=separation, y="separation", ax=axes[2]); axes[2].set_title("Separation"); sns.despine(ax=axes[2])
plt.tight_layout(); plt.show()

In [ ]:
# -----------------------------
# Metacell matrices
# -----------------------------
rna_ad.obs["SEACell"] = atac_ad.obs["SEACell"]

rna_ad.layers["counts"] = rna_ad.X
atac_ad.layers["counts"] = atac_ad.X
rna_meta_ad = SEACells.core.summarize_by_SEACell(rna_ad, SEACells_label="SEACell", summarize_layer="counts")
atac_meta_ad = SEACells.core.summarize_by_SEACell(atac_ad, SEACells_label="SEACell", summarize_layer="counts")

# Replace specific clone names in atac_ad.obs['clone2']
atac_ad.obs['clone3'] = atac_ad.obs['clone2']
atac_ad.obs['clone3'] = atac_ad.obs['clone3'].replace({
    'P06_clone_1_stress': 'P06_stress_Hi',
    'P06_clone_2_stress': 'P06_stress_Hi'
})
rna_ad.obs['clone3'] = atac_ad.obs['clone3']

top_ct = atac_ad.obs["clone2"].groupby(atac_ad.obs["SEACell"]).value_counts().groupby(level=0, group_keys=False).head(1)
rna_meta_ad.obs["clone2"] = top_ct[rna_meta_ad.obs_names].index.get_level_values(1)
atac_meta_ad.obs["clone2"] = top_ct[atac_meta_ad.obs_names].index.get_level_values(1)

top_ct = atac_ad.obs["clone3"].groupby(atac_ad.obs["SEACell"]).value_counts().groupby(level=0, group_keys=False).head(1)
rna_meta_ad.obs["clone3"] = top_ct[rna_meta_ad.obs_names].index.get_level_values(1)
atac_meta_ad.obs["clone3"] = top_ct[atac_meta_ad.obs_names].index.get_level_values(1)

# -----------------------------
# Save
# -----------------------------
dir_path = f"Reproducibility/Results/SEACells/TREKKER/"
os.makedirs(dir_path, exist_ok=True)

atac_ad.write(f'{dir_path}/UC_TREKKER_Malignant_atac_ad_w_SEACells.h5ad')
rna_ad.write(f'{dir_path}/UC_TREKKER_Malignant_rna_ad.h5ad')

rna_meta_ad.write(f"{dir_path}/UC_TREKKER_Malignant_rna_meta_ad.h5ad")
atac_meta_ad.write(f"{dir_path}/UC_TREKKER_Malignant_atac_meta_ad.h5ad")

rna_ad.obs.to_csv(f"{dir_path}/UC_TREKKER_Malignant_metadata_w_SEACells.txt", sep ='\t')
rna_meta_ad.obs.to_csv(f"{dir_path}/UC_TREKKER_Malignant_metacell_metadata.txt", sep="\t")